# NAFLD Fibrosis Prediction: Model Inference & Evaluation

> **Academic Research Prototype** — Not approved as a medical device or diagnostic instrument.

This notebook loads the fitted **Tuned Random Forest Model Bundle** (`research_model_bundle.joblib`), explores the 33 clinical predictors, evaluates the exploratory decision threshold (0.4158), and compares predictions against the clinical standard **FIB-4** index.

In [ ]:
# 1. Environment Setup & Repository Clone (for Google Colab or local Jupyter)
import os
import sys
from pathlib import Path

REPO_URL = 'https://github.com/shrfr/nafld-fibrosis-prediction.git'
REPO_DIR = 'nafld-fibrosis-prediction'

# If running in Colab and repo not yet cloned:
if not os.path.exists('data/NAFLD.xlsx') and not os.path.exists('../data/NAFLD.xlsx'):
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    %cd {REPO_DIR}

# Add src to Python path
for candidate in ['.', '..', 'nafld-fibrosis-prediction']:
    src_path = os.path.join(candidate, 'src')
    if os.path.exists(src_path) and src_path not in sys.path:
        sys.path.insert(0, os.path.abspath(src_path))
        break

print('Environment ready. Working directory:', os.getcwd())

In [ ]:
# 2. Load the Trained Model File
import joblib

candidate_model_paths = [
    'research_model_bundle.joblib',
    'models/research_model_bundle.joblib',
    '../outputs/final/research_model_bundle.joblib',
    'outputs/final/research_model_bundle.joblib',
    'notebooks/research_model_bundle.joblib'
]

model_path = next((p for p in candidate_model_paths if os.path.exists(p)), None)
if not model_path:
    raise FileNotFoundError('Model file research_model_bundle.joblib not found!')

bundle = joblib.load(model_path)
pipeline = bundle['pipeline']
features = bundle['features']
threshold = bundle['threshold']
best_params = bundle['best_parameters']

print(f' Successfully loaded trained model from: {model_path}')
print(f' Model Type: Tuned Random Forest Classifier ({len(features)} input features)')
print(f' Best Parameters: {best_params}')
print(f' Decision Threshold (Sensitivity >= 80%): {threshold:.4f}')

In [ ]:
# 3. Run Shared Setup to prepare Dataset (N=605) & 33 clean predictors
# Executing the setup cell
setup_path = 'setup_cell.py' if os.path.exists('setup_cell.py') else 'notebooks/setup_cell.py'
if os.path.exists(setup_path):
    with open(setup_path, 'r') as f:
        exec(f.read())
else:
    print('setup_cell.py not found locally; please run notebooks/setup_cell.py first.')

print(f'Cohort size: {len(X)} patients, {len(feats)} input features')
print(f'Significant Fibrosis (F >= 2) prevalence: {y.mean():.1%}')

In [ ]:
# 4. Model Predictions & Probability Estimation
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Predict continuous risk probabilities on cohort
y_probs = pipeline.predict_proba(X[features])[:, 1]
y_pred_tuned = (y_probs >= threshold).astype(int)
y_pred_default = (y_probs >= 0.50).astype(int)

# Evaluate discriminatory power
auc = roc_auc_score(y, y_probs)
print(f'Model Full-Cohort Apparent ROC-AUC: {auc:.4f}')

cm = confusion_matrix(y, y_pred_tuned)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f'At Tuned Research Threshold ({threshold:.4f}):')
print(f'  Sensitivity (Recall): {sensitivity:.1%}')
print(f'  Specificity:          {specificity:.1%}')
print(f'  Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}')

In [ ]:
# 5. Clinical Benchmark Comparison: Tuned RF vs. FIB-4
# FIB-4 cutoffs: <1.30 rule-out, >2.67 rule-in
fib4_scores = data['FIB4']
fib4_auc = roc_auc_score(y, fib4_scores)
fib4_ruleout_sens = ((fib4_scores >= 1.30) & (y == 1)).sum() / (y == 1).sum()
fib4_ruleout_spec = ((fib4_scores < 1.30) & (y == 0)).sum() / (y == 0).sum()

print(f'Clinical Benchmark Comparison:')
print(f'  Tuned RF AUC: {auc:.3f} vs FIB-4 AUC: {fib4_auc:.3f}')
print(f'  Tuned RF Sensitivity at {threshold:.2f}: {sensitivity:.1%} (Specificity: {specificity:.1%})')
print(f'  FIB-4 Sensitivity at 1.30 cutoff: {fib4_ruleout_sens:.1%} (Specificity: {fib4_ruleout_spec:.1%})')

In [ ]:
# 6. Single Patient Case Inference Example
# Pick a sample high-risk and sample low-risk patient
sample_patient = X[features].iloc[[0]].copy()
sample_prob = pipeline.predict_proba(sample_patient)[:, 1][0]
sample_pred = 'Significant Fibrosis Risk (F >= 2)' if sample_prob >= threshold else 'Low / Mild Fibrosis Risk (F 0-1)'

print('--- Sample Patient 0 ---')
print(f'Age: {sample_patient["Age"].values[0]}, AST: {sample_patient["AST"].values[0]}, ALT: {sample_patient["ALT"].values[0]}')
print(f'Predicted Probability: {sample_prob:.1%}')
print(f'Classification:        {sample_pred}')